# Chapter 3 (Hull) — Cross Hedging & Minimum-Variance Hedge Ratio 

In this continuation of Chapter 3, Hull explains how to hedge when:

- the asset you want to hedge is **not the same** as the futures underlying (**cross hedging**),
- the hedge is imperfect because spot and futures do not move identically,
- so we choose a hedge ratio that **minimizes the variance** (risk) of the hedged position.

We keep the same “Hull-level” approach:
1. Cross hedging (intuition)
2. Minimum-variance hedge ratio (formula + meaning)
3. Optimal number of futures contracts
4. (Optional) impact of daily settlement (one-day hedge)

## 3.4 Cross Hedging

**Cross hedging** occurs when the asset being hedged is **different** from the asset underlying the futures contract.

Example (Hull):
- An airline wants to hedge the future price of **jet fuel**.
- Jet fuel futures might be illiquid.
- The airline may use **heating oil futures** instead, because that market is more active.

Key idea:
- You accept that the hedge will not be perfect.
- You choose the hedge ratio to reduce risk as much as possible.

## Calculating the Minimum-Variance Hedge Ratio

Hull defines over the hedge horizon:

- $\Delta S$: change in the spot price of the asset being hedged  
- $\Delta F$: change in the futures price (contract used for hedging)

Hull assumes an approximately linear relation:

$$
\Delta S = a + b\,\Delta F + \varepsilon
$$

where:
- $a$ and $b$ are constants
- $\varepsilon$ is an error term (what futures cannot explain)

Let the hedge ratio be $h$ (how much futures we use per unit exposure).
The change in the **hedged** position is:

$$
\Delta S - h\,\Delta F
$$

Substitute the regression model:

$$
\Delta S - h\,\Delta F = a + (b-h)\Delta F + \varepsilon
$$

To minimize variance, Hull shows the best choice is:

$$
h^* = b
$$

And using the regression slope result:

$$
h^* = \rho\,\frac{\sigma_S}{\sigma_F}
$$

where:
- $\sigma_S$ = standard deviation of $\Delta S$
- $\sigma_F$ = standard deviation of $\Delta F$
- $\rho$ = correlation between $\Delta S$ and $\Delta F$

### Interpretation (very important)

- If $\rho$ is close to **1** (moves are strongly aligned), hedging can be very effective.
- If $\rho$ is small, the hedge will be weak (futures does not track spot well).
- If $\sigma_S$ is larger than $\sigma_F$, you usually need a hedge ratio larger than 1 (spot moves more).
- If $\sigma_F$ is larger, the hedge ratio may be less than 1.

In [1]:
import numpy as np

def min_variance_hedge_ratio(delta_S, delta_F):
    """
    Hull: h* = rho * sigma_S / sigma_F
    """
    delta_S = np.asarray(delta_S, dtype=float)
    delta_F = np.asarray(delta_F, dtype=float)

    sigma_S = delta_S.std(ddof=1)
    sigma_F = delta_F.std(ddof=1)
    rho = np.corrcoef(delta_S, delta_F)[0, 1]
    h_star = rho * (sigma_S / sigma_F)
    return h_star, rho, sigma_S, sigma_F

# Toy data: spot changes and futures changes
delta_F = np.array([0.021, 0.035, -0.046, 0.001, 0.044, -0.029, -0.026, -0.029, 0.048, -0.006, -0.036, -0.011, 0.019, -0.027, 0.029])
delta_S = np.array([0.029, 0.020, -0.044, 0.008, 0.026, -0.019, -0.010, -0.007, 0.043, 0.011, -0.036, -0.018, 0.009, -0.032, 0.023])

h_star, rho, sigma_S, sigma_F = min_variance_hedge_ratio(delta_S, delta_F)

print(f"rho     = {rho:.3f}")
print(f"sigma_S = {sigma_S:.4f}")
print(f"sigma_F = {sigma_F:.4f}")
print(f"h*      = {h_star:.2f}")

rho     = 0.928
sigma_S = 0.0263
sigma_F = 0.0313
h*      = 0.78


## Optimal Number of Futures Contracts

Hull defines:
- $Q_A$ = size of the position being hedged (in units of the asset)
- $Q_F$ = size of one futures contract (units per contract)
- $h^*$ = optimal hedge ratio (minimum-variance)

Then the optimal number of futures contracts is:

$$
N^* = \frac{h^* Q_A}{Q_F}
$$

In [2]:
# Hull-like example
Q_A = 2_000_000     # jet fuel exposure (gallons)
Q_F = 42_000        # heating oil contract size (gallons)
h_star = 0.78       # from the example-style calculation

N_star = (h_star * Q_A) / Q_F
print(f"N* = {N_star:.2f} -> rounded = {round(N_star)} contracts")

N* = 37.14 -> rounded = 37 contracts
